# VibeML: Personalized Spotify Playlist Agent

This notebook defines the VibeML agent, a personalized Spotify playlist generation agent that combines a user's listening history with real-time mood and activity input to build a playlist for the current moment.

##Install Dependencies

In [0]:
# Install dependencies
%pip install spotipy

In [0]:
%restart_python

## Imports and Setup

Import all necessary libraries and set up the environment for the agent.

In [0]:
# Core libraries
import os
import json
import pandas as pd

# Spotify API
import spotipy
from spotipy.oauth2 import SpotifyOAuth

# Databricks / MLflow
import mlflow

# LLM - we will fill this in once we confirm which LLM we are using

### Spotify Authentication

This section sets up the Spotify client using the full OAuth flow, which gives the agent access to user specific data including liked songs, playlists, listening history, and the ability to save playlists.

#### Setup Steps (first time only)

1. Go to developer.spotify.com and log in with your Spotify account
2. Click "Create app" and fill in the following:
   - App name: VibeML
   - App description: Personalized Spotify Playlist Agent
   - Redirect URI: http://127.0.0.1:8080
   - Check the "Web API" box
3. Once the app is created click "Settings" and copy your Client ID and Client Secret
4. Run the credentials cell below and enter your Client ID and Client Secret when prompted
5. Run this cell, it will print a Spotify authorization URL in the output:
    e.g. with the client ID and key as your specific credentials : "https://accounts.spotify.com/authorize?client_id=e883018d172a497a94c2e78f9b300117&response_type=code&redirect_uri=http%3A%2F%2F127.0.0.1%3A8080&scope=user-library-read+user-read-recently-played+playlist-read-private+playlist-modify-public+playlist-modify-private+user-read-currently-playing"
    
6. Copy that URL and paste it into your browser
7. Log in with your Spotify account and click "Agree"
8. You will be redirected to a URL starting with http://127.0.0.1:8080/?code=
9. Copy that full URL from your browser address bar
10. Paste it into the notebook where it says "Enter the URL you were redirected to:" and hit enter
11. You should see "Spotify connected successfully" with your display name

In [0]:
# CREDENTIALS CELL
import os

# Set Spotify credentials manually at runtime
# These are never stored in the notebook or GitHub
os.environ["SPOTIFY_CLIENT_ID"] = dbutils.widgets.get("client_id") if False else input("Enter your Spotify Client ID: ")
os.environ["SPOTIFY_CLIENT_SECRET"] = dbutils.widgets.get("client_secret") if False else input("Enter your Spotify Client Secret: ")
os.environ["SPOTIFY_REDIRECT_URI"] = "http://127.0.0.1:8080"

In [0]:
# AUTHENTICATION CELL
from spotipy.oauth2 import SpotifyClientCredentials

# Set up the Spotify client using client credentials
sp = spotipy.Spotify(auth_manager=SpotifyClientCredentials(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"]
))

# Test the connection
results = sp.search(q="test", limit=1)
print("Spotify connected successfully")
print(f"Test track: {results['tracks']['items'][0]['name']}")

In [0]:
from spotipy.oauth2 import SpotifyOAuth

# Define the scopes we need from the Spotify API
SPOTIFY_SCOPES = " ".join([
    "user-library-read",
    "user-read-recently-played",
    "playlist-read-private",
    "playlist-modify-public",
    "playlist-modify-private",
    "user-read-currently-playing"
])

# Set up the Spotify OAuth client
sp = spotipy.Spotify(auth_manager=SpotifyOAuth(
    client_id=os.environ["SPOTIFY_CLIENT_ID"],
    client_secret=os.environ["SPOTIFY_CLIENT_SECRET"],
    redirect_uri=os.environ["SPOTIFY_REDIRECT_URI"],
    scope=SPOTIFY_SCOPES,
    open_browser=False
))

# Test the connection
user = sp.current_user()
print(f"Spotify connected successfully")
print(f"Logged in as: {user['display_name']}")

## Tool Definitions

The following functions represent the tools the agent can call during its reasoning process.

### get_user_profile
- Pulls the user's liked songs, saved playlists, and listening history from the Spotify API.

In [0]:
def get_user_profile(sp):
    """
    Pulls the user's liked songs, saved playlists, and listening history from the Spotify API.
    
    Args:
        sp: Authenticated Spotipy client
    Returns:
        dict: User's liked songs, playlists, and listening history
    """
    pass

### get_currently_playing
- Fetches the track the user is currently listening to from the Spotify API.


In [0]:
def get_currently_playing(sp):
    """
    Fetches the track the user is currently listening to from the Spotify API.
    
    Args:
        sp: Authenticated Spotipy client
    Returns:
        dict: Currently playing track info
    """
    pass

### search_catalog
- Searches the Spotify catalog to verify a track exists and is available.

In [0]:
def search_catalog(sp, track_name=None, artist_name=None):
    """
    Searches the Spotify catalog to verify a track exists and is available.
    
    Args:
        sp: Authenticated Spotipy client
        track_name (str): Name of the track to search for
        artist_name (str): Name of the artist
    Returns:
        dict: Track info if found, None if not found
    """
    pass

### search_songs
- Queries the processed Kaggle dataset by audio feature ranges to find matching tracks.

In [0]:
def search_songs(energy=None, valence=None, tempo=None, danceability=None, speechiness=None, instrumentalness=None):
    """
    Queries the Spotify Tracks Dataset (114,000 tracks across 125 genres) by audio 
    feature ranges to find matching tracks.
    
    Args:
        energy (tuple): Min and max energy range e.g. (0.7, 1.0)
        valence (tuple): Min and max valence range
        tempo (tuple): Min and max tempo range
        danceability (tuple): Min and max danceability range
        speechiness (tuple): Min and max speechiness range
        instrumentalness (tuple): Min and max instrumentalness range
    Returns:
        DataFrame: Tracks matching the specified audio feature ranges
    """
    pass

### trend_filter
- Queries the second Kaggle dataset to narrow recommendations by release era or artist popularity.


In [0]:
def trend_filter(era=None, popularity=None):
    """
    Queries the Spotify Global Music Dataset (2009-2025) to narrow recommendations 
    by release era or artist popularity.
    
    Args:
        era (tuple): Min and max release year range e.g. (2009, 2015)
        popularity (tuple): Min and max artist popularity range
    Returns:
        DataFrame: Tracks matching the specified trend filters
    """
    pass

### vibe_mapping
- Takes the user's natural language mood or activity description and uses the LLM to translate it into target audio feature ranges.

In [0]:
def vibe_mapping(user_input):
    """
    Takes the user's natural language mood or activity description and uses the LLM 
    to translate it into target audio feature ranges.
    
    Args:
        user_input (str): User's natural language description of their mood or activity
    Returns:
        dict: Target audio feature ranges derived from the user's input
    """
    pass

### feedback_refinement
- Adjusts audio feature target ranges based on the user's tester song feedback.



In [0]:
def feedback_refinement(accepted_tracks, rejected_tracks, current_ranges):
    """
    Adjusts audio feature target ranges based on the user's tester song feedback.
    
    Args:
        accepted_tracks (list): Track IDs the user accepted
        rejected_tracks (list): Track IDs the user rejected
        current_ranges (dict): Current audio feature target ranges
    Returns:
        dict: Updated audio feature ranges
    """
    pass

### save_playlist
- Saves the final list of tracks as a new playlist to the user's Spotify account.

In [0]:
def save_playlist(sp, track_ids, playlist_name):
    """
    Saves the final list of tracks as a new playlist to the user's Spotify account.
    
    Args:
        sp: Authenticated Spotipy client
        track_ids (list): List of Spotify track IDs to add to the playlist
        playlist_name (str): Name of the playlist to create
    Returns:
        str: URL of the created playlist
    """
    pass

## Agent Definition

This section defines the agent's system prompt, registers the tools, and sets up the LLM. The agent uses a ReAct style reasoning pattern, that is it reasons over the user's input, decides which tools to call, observes the results, and reasons again until it has enough information to build the final playlist.

In [0]:
# System prompt that tells the LLM what its job is
SYSTEM_PROMPT = """
You are VibeML, a personalized Spotify playlist agent. Your job is to help the user build a playlist that fits exactly what they are feeling or doing right now.

You have access to the following tools:
- get_user_profile: Pull the user's liked songs, playlists, and listening history from Spotify
- get_currently_playing: Fetch the track the user is currently listening to
- search_catalog: Verify a song exists and is available on Spotify
- search_songs: Search for songs by audio feature ranges like energy, valence, tempo, and danceability
- trend_filter: Filter songs by release era or artist popularity
- vibe_mapping: Translate the user's mood or activity description into target audio feature ranges
- feedback_refinement: Adjust audio feature ranges based on the user's tester song feedback
- save_playlist: Save the final playlist to the user's Spotify account

Follow these steps when helping a user:
1. Pull the user's profile to understand their taste
2. Ask about their current mood, activity, and desired vibe
3. Use vibe_mapping to translate their input into audio feature ranges
4. Search for matching songs using search_songs and trend_filter as needed
5. Present 3 tester songs and collect feedback
6. Use feedback_refinement to adjust if needed
7. Build and save the final playlist

Only help with music playlist requests. Politely decline anything outside of that scope.
"""

# Tool list - full implementation will be added in the next phase
TOOLS = [
    get_user_profile,
    get_currently_playing,
    search_catalog,
    search_songs,
    trend_filter,
    vibe_mapping,
    feedback_refinement,
    save_playlist
]

print("Agent definition loaded successfully")

## LLM Setup

This section configures the LLM that powers the agent's reasoning

In [0]:
from openai import OpenAI
import mlflow

# LLM model configuration
LLM_MODEL = "databricks-meta-llama-3-3-70b-instruct" # Can be changed, just an example

# Get the workspace URL and token automatically from the Databricks environment
DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
DATABRICKS_HOST = spark.conf.get("spark.databricks.workspaceUrl")

# Set up the Databricks OpenAI compatible client
client = OpenAI(
    api_key=DATABRICKS_TOKEN,
    base_url=f"https://{DATABRICKS_HOST}/serving-endpoints"
)

# Test that the model endpoint is accessible
def test_llm_setup():
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Hello, can you help me build a playlist?"}
        ]
    )
    print("LLM setup successful")
    print(f"Model: {LLM_MODEL}")
    print(f"Response: {response.choices[0].message.content}")

test_llm_setup()

## Test Run

A simple test to verify the agent setup is in place and the tools are registered correctly. Full end to end testing will be completed once the LLM and Spotify API authentication are configured.

In [0]:
# Test that all tools are defined and accessible
def test_agent_setup():
    print("Testing agent setup...\n")
    
    # Check all tools are defined
    for tool in TOOLS:
        print(f"Tool registered: {tool.__name__}")
    
    # Check system prompt is loaded
    print("\nSystem prompt loaded successfully")
    print(f"Total tools registered: {len(TOOLS)}")
    print("\nAgent setup test complete.")

test_agent_setup()